In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt

import sys
import os

sys.path.append(os.path.abspath('..'))
print(sys.path)

from training.target_flow_lib import TargetCompositeFlow

In [ ]:

def analyze_target_flow():
    # 1. Setup Configuration
    # Use CPU for analysis to avoid CUDA init overhead if just debugging
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    batch_size = 1024
    img_size = 32
    D = 3 * img_size * img_size  # Total dimensions
    
    noise_seed = 42 
    torch.manual_seed(noise_seed)
    
    print(f"Loading CIFAR-10 on {device}...")
    
    
    # Define the exact statistics from CSFMcode/datasets.py
    stats = {
        'mean': [0.4914, 0.4822, 0.4465],
        'std': [0.2470, 0.2435, 0.2616]
    }
    
    try:
        # Combine ToTensor with Normalize
        transform = torchvision.transforms.Compose([
            torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize(**stats)
        ])
        
        dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, 
                                             transform=transform)
    except:
        print("Could not download CIFAR. Using random noise (already N(0,1)).")
        dataset = None

    if dataset:
        loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
        x1, _ = next(iter(loader))
    else:
        x1 = torch.randn(batch_size, 3, img_size, img_size)

    x1 = x1.to(device)
    
    # 3. Initialize the Physics Engine (New Implementation)
    flow_engine = TargetCompositeFlow(img_shape=(3, img_size, img_size), device=device)
    
    # 4. Sweep through Time t
    # Draft implies t=0 (Clean) -> t=1 (Degraded/Noisy)
    time_steps = torch.linspace(0, 1, 50)
    
    # Metrics storage
    vel_norms = []      # Expect approx sqrt(D) ~ sqrt(3072) ~ 55.4
    vel_stds = []       # Expect approx 1.0 (Unit Variance per pixel)
    state_norms = []    # Expect approx sqrt(D) if signal is preserved
    
    # print(f"\nAnalyzing Flow (K={flow_engine.K}, Scaling=sqrt({2*flow_engine.K}))...")
    # print(f"Theoretical Norm Target: ~{np.sqrt(D):.2f} (sqrt(D))")
    print("-" * 65)
    print(f"{'Time':<10} | {'Vel Std':<10} | {'Vel Norm':<10} | {'State Norm':<10}")
    print("-" * 65)
    
    for t_val in time_steps:
        # Create time batch [B, 1, 1, 1]
        t = torch.ones(batch_size, 1, 1, 1).to(device) * t_val
        
        # Compute Target
        with torch.no_grad():
            x_t, u_t = flow_engine.compute_target(x1, t)
        
        # --- METRIC 1: Velocity Standard Deviation ---
        # The draft says we scale by sqrt(2K) to get "Unitary Variance".
        # This means the standard deviation of the pixels in u_t should be close to 1.0.
        v_std = u_t.std().item()
        vel_stds.append(v_std)
        
        # --- METRIC 2: Velocity Magnitude (Global Norm) ---
        # L2 norm per image. If std is 1, norm should be sqrt(D).
        u_flat = u_t.reshape(batch_size, -1)
        v_norm = torch.norm(u_flat, dim=1).mean().item()
        vel_norms.append(v_norm)
        
        # --- METRIC 3: State Energy ---
        # Check if Local VP holds (Variance Preservation)
        x_flat = x_t.reshape(batch_size, -1)
        s_norm = torch.norm(x_flat, dim=1).mean().item()
        state_norms.append(s_norm)
        
        if t_val in [0.0, 0.5, 1.0]: 
            print(f"{t_val.item():.2f}       | {v_std:.4f}     | {v_norm:.4f}     | {s_norm:.4f}")

    # 5. Visualization
    plt.figure(figsize=(15, 5))
    
    # Plot A: Velocity Standard Deviation (Should be ~1.0)
    plt.subplot(1, 3, 1)
    plt.plot(time_steps, vel_stds, label='Actual Std', color='blue', linewidth=2)
    plt.axhline(y=1.0, color='r', linestyle='--', label='Target (1.0)')
    plt.title('Velocity Standard Deviation (per pixel)')
    plt.xlabel('Time t')
    plt.ylabel('Std')
    plt.legend()
    # plt.yscale('log')
    plt.grid(True, alpha=0.3)
    
    # Plot B: Velocity Norm (Should be ~sqrt(D))
    plt.subplot(1, 3, 2)
    plt.plot(time_steps, vel_norms, label='||v|| (Avg Norm)', color='purple')
    plt.axhline(y=np.sqrt(D), color='r', linestyle='--', label=f'sqrt(D)={np.sqrt(D):.1f}')
    plt.title(f'Velocity Magnitude (Expect ~{np.sqrt(D):.0f})')
    plt.xlabel('Time t')
    plt.grid(True, alpha=0.3)
    # plt.yscale('log')
    plt.legend()

    # Plot C: State Evolution
    plt.subplot(1, 3, 3)
    plt.plot(time_steps, state_norms, label='||x_t||', color='green')
    plt.title('State Norm Evolution (VP Check)')
    plt.xlabel('Time t')
    # plt.yscale('log')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('flow_analysis_check.png')
    print("\nAnalysis complete. Check 'flow_analysis_check.png'.")
    print("If 'Vel Std' stays near 1.0, your scaling C=sqrt(2K) is correct.")

if __name__ == "__main__":
    analyze_target_flow()

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np
# from target_flow_lib import TargetCompositeFlow

def unnormalize(tensor, stats):
    """Reverts Z-Score normalization for visualization."""
    mean = torch.tensor(stats['mean']).view(1, 3, 1, 1).to(tensor.device)
    std = torch.tensor(stats['std']).view(1, 3, 1, 1).to(tensor.device)
    x = tensor * std + mean
    return torch.clamp(x, 0, 1)

def normalize_for_vis(tensor):
    """
    Normalizes a derivative tensor (with negative values) to [0, 1] 
    for visualization purposes. 
    Gray (0.5) = Zero change. Black/White = Strong negative/positive change.
    """
    # Robust min-max normalization per image
    flat = tensor.flatten(1)
    # Using 1% and 99% quantiles is more robust to outliers than min/max
    low = torch.quantile(flat, 0.01, dim=1, keepdim=True).unsqueeze(-1).unsqueeze(-1)
    high = torch.quantile(flat, 0.99, dim=1, keepdim=True).unsqueeze(-1).unsqueeze(-1)
    
    # Avoid division by zero
    diff = high - low
    diff[diff < 1e-5] = 1.0
    
    return torch.clamp((tensor - low) / diff, 0, 1)

def visualize_complete_evolution():
    # 1. Setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running visualization on {device}...")
    
    stats = {
        'mean': [0.4914, 0.4822, 0.4465],
        'std': [0.2470, 0.2435, 0.2616]
    }
    
    # 2. Load Data
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(**stats)
    ])
    
    try:
        dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
        loader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True) # Reduced batch size for layout
        real_batch, _ = next(iter(loader))
    except:
        print("CIFAR-10 not found. Using random noise.")
        real_batch = torch.randn(4, 3, 32, 32)
        
    real_batch = real_batch.to(device)
    
    # 3. Physics Engine
    flow_engine = TargetCompositeFlow(img_shape=(3, 32, 32), device=device)
    
    # 4. Generate Trajectories
    time_steps = [0.0, 0.2, 0.5, 0.8, 1.0]
    
    trajectory_x = [] # States
    trajectory_u = [] # Velocities
    
    noise_seed = 42 
    
    print("Computing flow trajectories...")
    for t_val in time_steps:
        # Force fixed noise for smooth evolution
        torch.manual_seed(noise_seed)
        
        t = torch.ones(real_batch.shape[0], 1, 1, 1).to(device) * t_val
        
        with torch.no_grad():
            x_t, u_t = flow_engine.compute_target(real_batch, t)
            
        # Un-normalize State (to see image)
        x_vis = unnormalize(x_t, stats)
        trajectory_x.append(x_vis.cpu())
        
        # Normalize Velocity (to see structure)
        u_vis = normalize_for_vis(u_t)
        trajectory_u.append(u_vis.cpu())

    # 5. Plotting (Interleaved Rows)
    num_samples = real_batch.shape[0]
    num_times = len(time_steps)
    
    # Grid: 2 rows per sample (State + Velocity)
    fig, axes = plt.subplots(num_samples * 2, num_times, figsize=(num_times * 2.5, num_samples * 4))
    
    for samp in range(num_samples):
        row_x = samp * 2
        row_u = samp * 2 + 1
        
        for col in range(num_times):
            # --- Plot State x_t ---
            ax_x = axes[row_x, col]
            img_x = trajectory_x[col][samp].permute(1, 2, 0).numpy()
            ax_x.imshow(img_x)
            ax_x.axis('off')
            
            # Labels
            if samp == 0: ax_x.set_title(f"t = {time_steps[col]:.1f}", fontsize=12)
            if col == 0: ax_x.text(-5, 16, f"State\nImg {samp}", va='center', ha='right', fontsize=10, weight='bold')

            # --- Plot Velocity u_t ---
            ax_u = axes[row_u, col]
            img_u = trajectory_u[col][samp].permute(1, 2, 0).numpy()
            ax_u.imshow(img_u) # cmap='RdBu' doesn't apply to RGB, so we show the normalized RGB structure
            ax_u.axis('off')
            
            if col == 0: ax_u.text(-5, 16, f"Velocity\nTarget", va='center', ha='right', fontsize=10, style='italic')

    plt.tight_layout()
    plt.savefig("flow_trajectory_with_vectors.png")
    print("Saved visualization to 'flow_trajectory_with_vectors.png'")
    plt.show()

if __name__ == "__main__":
    visualize_complete_evolution()

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt
import numpy as np
# from target_flow_lib import TargetCompositeFlow

def unnormalize(tensor, stats):
    """Reverts Z-Score normalization for visualization."""
    mean = torch.tensor(stats['mean']).view(1, 3, 1, 1).to(tensor.device)
    std = torch.tensor(stats['std']).view(1, 3, 1, 1).to(tensor.device)
    x = tensor * std + mean
    return torch.clamp(x, 0, 1)

def visualize_diverging_velocity():
    # 1. Setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running diverging visualization on {device}...")
    
    stats = {
        'mean': [0.4914, 0.4822, 0.4465],
        'std': [0.2470, 0.2435, 0.2616]
    }
    
    # 2. Load Data
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
        torchvision.transforms.Normalize(**stats)
    ])
    
    try:
        dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
        loader = torch.utils.data.DataLoader(dataset, batch_size=4, shuffle=True)
        real_batch, _ = next(iter(loader))
    except:
        print("CIFAR-10 not found. Using random noise.")
        real_batch = torch.randn(4, 3, 32, 32)
        
    real_batch = real_batch.to(device)
    
    # 3. Physics Engine
    flow_engine = TargetCompositeFlow(img_shape=(3, 32, 32), device=device)
    
    # 4. Generate Trajectories
    time_steps = [0.0, 0.05, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
    
    trajectory_x = [] 
    trajectory_u = [] 
    
    # Fixed seed for consistent noise across time steps
    noise_seed = 42 
    
    print("Computing flow trajectories...")
    for t_val in time_steps:
        torch.manual_seed(noise_seed)
        
        t = torch.ones(real_batch.shape[0], 1, 1, 1).to(device) * t_val
        
        with torch.no_grad():
            x_t, u_t = flow_engine.compute_target(real_batch, t)
            
        # A. State (Image)
        x_vis = unnormalize(x_t, stats)
        trajectory_x.append(x_vis.cpu())
        
        # B. Velocity (Heatmap Preparation)
        # We take the mean across channels [C, H, W] -> [H, W]
        # This shows the "net intensity change" per pixel.
        u_mean = u_t.mean(dim=1) 
        trajectory_u.append(u_mean.cpu())

    # 5. Plotting
    num_samples = real_batch.shape[0]
    num_times = len(time_steps)
    
    fig, axes = plt.subplots(num_samples * 2, num_times, figsize=(num_times * 3, num_samples * 4))
    
    for samp in range(num_samples):
        row_x = samp * 2
        row_u = samp * 2 + 1
        
        # Calculate consistent color scale for this sample's velocity row
        # (This allows us to see the magnitude fading or growing relative to the whole path)
        # Or we can scale per image. Let's scale per image to see local contrast.
        
        for col in range(num_times):
            # --- Plot State x_t ---
            ax_x = axes[row_x, col]
            img_x = trajectory_x[col][samp].permute(1, 2, 0).numpy()
            ax_x.imshow(img_x)
            ax_x.axis('off')
            
            if samp == 0: ax_x.set_title(f"t = {time_steps[col]:.1f}", fontsize=12)
            if col == 0: ax_x.text(-5, 16, f"State\nImg {samp}", va='center', ha='right', fontsize=10, weight='bold')

            # --- Plot Velocity u_t (Diverging) ---
            ax_u = axes[row_u, col]
            
            vel_map = trajectory_u[col][samp].numpy()
            
            # Symmetric scaling ensures 0 is always White/Center
            max_val = max(abs(vel_map.max()), abs(vel_map.min())) + 1e-5
            
            # cmap='RdBu_r': Red is positive (Right/Up), Blue is negative (Left/Down)
            im = ax_u.imshow(vel_map, cmap='RdBu_r', vmin=-max_val, vmax=max_val)
            ax_u.axis('off')
            
            # Optional: Add colorbar to the last column to show scale
            if col == num_times - 1:
                cbar = plt.colorbar(im, ax=ax_u, fraction=0.046, pad=0.04)
                cbar.ax.tick_params(labelsize=8)
            
            if col == 0: ax_u.text(-5, 16, f"Velocity\n(Red+, Blu-)", va='center', ha='right', fontsize=10, style='italic')

    plt.tight_layout()
    plt.savefig("flow_diverging_velocity.png")
    print("Saved visualization to 'flow_diverging_velocity.png'")
    plt.show()

if __name__ == "__main__":
    visualize_diverging_velocity()